# Protoplanetary-Disk Denoising — Unified Research Notebook (Week 1 → Week 4)

**Krishan Yadav · ML4Sci EXXA · GSoC 2026**

This single notebook merges the complete research journey from five working notebooks
(`01_data_exploration` → `05_diffusion_kaggle`) **without losing information**. Every model
architecture, training procedure, evaluation metric, finding and visualization is preserved.
It is fully executable top-to-bottom on Kaggle.

## Research Goal
Denoise interferometric (ALMA-like) observations of protoplanetary disks — recovering ring/gap
structure from noisy "dirty" images — and compare classical filters against four ML approaches
(Autoencoder → VAE → U-Net → Diffusion), with **SSIM** as the primary structure-fidelity metric.

## Dataset Description
975 paired images, `600×600`, normalised to `[0, 1]`.
`dirty.npy` (noisy, originally float64, 2.7 GB) and `clean.npy` (ground-truth simulation, float32, 1.4 GB).

## The journey at a glance
| Week | Milestone | Section |
|---|---|---|
| 1–2 | Data EDA + classical baselines (Gaussian/Median/Wiener) | §Data, §Baselines |
| 2–3 | Denoising **Autoencoder** (MSE → Hybrid MSE+SSIM) | §Autoencoder |
| 3 | **VAE** (probabilistic latent, +KL) | §VAE |
| 3 | Supervised **U-Net** (skip-connections, DDPM backbone) | §U-Net |
| 4 | Conditional **DDPM** diffusion (DDIM, multi-GPU) | §Diffusion |


## Phase 1 — Project Audit (notebook inventory)

| Notebook | Purpose | Key components | Dependencies | Important findings |
|---|---|---|---|---|
| 01_data_exploration | EDA + classical baselines | `run_baselines`, bar chart, sample grid | `src.baselines`, scipy, skimage | 975×600×600, [0,1]; dirty=float64→cast; **Gaussian σ=2 best classical** (SSIM 0.521) |
| 02_autoencoder | AE: MSE → Hybrid loss | `DenoisingAutoencoder`, `PatchDataset`, `HybridLoss`, torchinfo | `src.models.autoencoder`, `src.utils.losses` | 1.73M params; MSE-only val 0.007239 @ep26; **Hybrid MSE 0.007181 beats MSE-only** |
| 03_vae | VAE + first unified eval | `DenoisingVAE`, `VAELoss`, grad-accum, sliding-window eval | `src.models.vae` | latent=128, KL γ=0.001; AE-Hybrid tops table at **SSIM 0.7609** |
| 04_unet | Supervised U-Net + 8-method leaderboard | `DenoisingUNet` (t=0, sigmoid) | `src.models.unet` | 3.4M; U-Net SSIM 0.7044; skip-conn ready as DDPM ε-backbone |
| 05_diffusion | Conditional DDPM (Kaggle) | `DiffusionUNet`, `DenoisingDiffusion`, DDIM, EMA, multi-GPU | `src.training.diffusion` | scaled 17.2M; local 40-ep weak (SSIM 0.235) → needs long Kaggle training |

### Cell-action ledger
| Action | What | Justification |
|---|---|---|
| **Preserve** | all 4 architectures, all training loops, all findings, all eval protocols, all figures | core research record — shown here (architectures rendered from the real `src/` source via `inspect.getsource`) |
| **Merge** | 5 bootstrap cells → 1; 4 `PatchDataset` copies → 1; 2 sliding-window evaluators → 1 | byte-identical duplicates |
| **Refactor** | scattered `SEED/PATCH/BATCH/EPOCHS` → one `CONFIG`; `../data`,`../results` → Kaggle-resolved paths | Phase-5 Kaggle optimisation |
| **Remove** | `02` cell 30 (200-line `__main__` script) | redundant re-implementation of the AE-Hybrid training already in `02` cells 12/25; logic preserved in the consolidated trainer, recorded numbers kept in markdown |
| **Remove** | repeated `import` / `device` / `sys.path` re-setup cells | replaced by one global Environment Setup; no information lost |


## Phase 2 — Dependency Analysis

**Duplicates found & resolved**
- *Imports*: each notebook re-imported numpy/torch/plt and re-ran `sys.path.insert` → **one** Imports cell.
- *`PatchDataset`*: defined 4× (nb02 ×2, nb03, nb04), all identical (random 64×64 crop + per-patch min-max) → **one** canonical class.
- *Sliding-window inference* (`nn_denoise`) + metric helper (`calc`): defined in nb03 and nb04 → **one** evaluator.
- *Loaders*: rebuilt per notebook → **one** supervised loader pair + one diffusion (2-channel) loader pair.
- *Training loops*: AE/VAE/U-Net loops shared the same skeleton → kept distinct (different loss signatures) but driven by one `CONFIG`.

**Execution dependency graph (top-to-bottom, no hidden state):**
```
        Data (dirty.npy, clean.npy)
                  |
          Preprocessing (per-patch min-max norm)
                  |
        Dataset / DataLoaders  ──────────────┐
                  |                           |
        Classical Baselines                  |
                  |                           |
        Autoencoder  ──►  VAE  ──►  U-Net     |  (share loaders, CONFIG, evaluator)
                  |        |         |        |
                  └────────┴────►  Unified Evaluation
                                      |
                            Diffusion (DDPM)  ◄── 2-channel loaders
                                      |
                            Visualization & Final Pipeline
```
Resolved conflict: the diffusion model needs `[dirty, clean]` 2-channel patches in `[-1,1]`
(via `create_dataloaders`), whereas AE/VAE/U-Net use single-channel `[0,1]` patches — both are
built explicitly so neither overwrites the other.

## Phase 3 — How to read this notebook
Each model section opens with an **Evolution** callout (▶ *Previous / New / Why / Benefit*) so the
notebook reads as a research narrative, not a script dump.


# Environment Setup


### Imports — Kaggle bootstrap
Clones the fork for `src/` (single source of truth for model code), installs deps, links the
uploaded data Dataset, and puts `src/` on the path. No-op off Kaggle.

**Before running on Kaggle:** Accelerator = *GPU T4 ×2*, Internet = *On*, then *Add Input* → your
`dirty.npy`/`clean.npy` Dataset.


In [ ]:
import os, sys, subprocess, glob

ON_KAGGLE = os.path.exists('/kaggle')
if ON_KAGGLE:
    REPO_URL = 'https://github.com/KrishanYadav333/EXXA.git'
    BRANCH   = 'week-4'
    REPO     = '/kaggle/working/EXXA'
    PKG      = os.path.join(REPO, 'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',REPO_URL,REPO], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','pytorch-msssim','torchinfo'], check=True)
    # link uploaded Kaggle Dataset as <repo>/data
    os.makedirs(os.path.join(PKG,'data'), exist_ok=True)
    hits = glob.glob('/kaggle/input/**/dirty.npy', recursive=True)
    if hits:
        sd = os.path.dirname(hits[0])
        for fn in ('dirty.npy','clean.npy'):
            dst = os.path.join(PKG,'data',fn)
            if not os.path.exists(dst):
                try: os.symlink(os.path.join(sd,fn), dst)
                except OSError:
                    import shutil; shutil.copy(os.path.join(sd,fn), dst)
    else:
        print('WARNING: dirty.npy not under /kaggle/input — use *Add Input* to attach your data.')
    os.chdir(os.path.join(PKG,'notebooks'))
    if PKG not in sys.path: sys.path.insert(0, PKG)
else:
    # local: ensure repo root (one level up from notebooks/) is importable
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'):
        os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
print('cwd :', os.getcwd())

In [ ]:
import time, inspect, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# classical baselines + metrics
from scipy.ndimage import gaussian_filter, median_filter
from scipy.signal import wiener as wiener_filter
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
from pytorch_msssim import ssim as ssim_torch

# project code (the real architectures live in src/)
from src.baselines import gaussian_denoise, median_denoise, wiener_denoise, run_baselines
from src.models.autoencoder import DenoisingAutoencoder
from src.models.vae import DenoisingVAE
from src.models.unet import DenoisingUNet
from src.models.diffusion_unet import DiffusionUNet, default_diffusion_config
from src.models.noise_scheduler import NoiseScheduler
from src.utils.losses import HybridLoss, VAELoss
from src.training.diffusion import DenoisingDiffusion
import src.training.diffusion as diffusion_mod
from src.data.dataset import create_dataloaders
print('imports OK')

### Configuration
One global `CONFIG` controls every stage. Flip `QUICK_TEST=True` for a fast top-to-bottom smoke run.


In [ ]:
CONFIG = {
    'SEED'        : 42,
    'PATCH_SIZE'  : 64,
    'BATCH_SIZE'  : 16,       # supervised models
    'GRAD_ACCUM'  : 4,        # effective batch 64
    'LR'          : 1e-3,     # supervised
    # epochs per stage (bump for stronger results; diffusion dominates runtime)
    'EPOCHS_AE_MSE'  : 30,
    'EPOCHS_AE_HYB'  : 30,
    'EPOCHS_VAE'     : 30,
    'EPOCHS_UNET'    : 30,
    'EPOCHS_DIFF'    : 150,
    'DIFF_LR'        : 2e-5,
    'DIFF_BATCH_IMG' : 16,    # diffusion images/batch (x PATCH_N patches)
    'DIFF_PATCH_N'   : 4,
    'DDIM_STEPS'     : 50,    # plan Task 4.1 (~50-step DDIM)
    # evaluation
    'N_EVAL_IMAGES'  : 100,   # full-image sliding-window leaderboard
    'STRIDE'         : 32,
    'BATCH_INFER'    : 32,
    'N_EVAL_PATCHES' : 256,   # shared-patch unified comparison (incl. DDPM)
    # behaviour
    'QUICK_TEST'     : False, # True -> 2 epochs / tiny eval for a fast end-to-end check
    'RESUME_DIFF'    : True,  # resume diffusion from checkpoint if present
}

if CONFIG['QUICK_TEST']:
    for k in ['EPOCHS_AE_MSE','EPOCHS_AE_HYB','EPOCHS_VAE','EPOCHS_UNET','EPOCHS_DIFF']:
        CONFIG[k] = 2
    CONFIG['N_EVAL_IMAGES'] = 10
    CONFIG['N_EVAL_PATCHES'] = 32

CKPT_DIR = '/kaggle/working/checkpoints' if ON_KAGGLE else '../results/checkpoints'
OUT_DIR  = '/kaggle/working' if ON_KAGGLE else '../results'
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
print('checkpoints ->', CKPT_DIR)
print('outputs     ->', OUT_DIR)

### Reproducibility


In [ ]:
def seed_everything(seed=CONFIG['SEED']):
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()
print('seeded with', CONFIG['SEED'])

### GPU Setup
Automatic device + multi-GPU detection (the DDPM uses `nn.DataParallel` across both T4s).


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPU  = torch.cuda.device_count()
print('device :', device, '| GPUs:', N_GPU)
if N_GPU:
    for i in range(N_GPU):
        p = torch.cuda.get_device_properties(i)
        print(f'  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB')

# Data Loading


In [ ]:
DATA_DIR = '../data'
dirty_all = np.load(os.path.join(DATA_DIR,'dirty.npy')).astype(np.float32)   # cast float64->float32
clean_all = np.load(os.path.join(DATA_DIR,'clean.npy')).astype(np.float32)
print('dirty', dirty_all.shape, dirty_all.dtype, '| clean', clean_all.shape, clean_all.dtype)

## Exploratory Data Analysis


In [ ]:
seed_everything()
ids = np.random.default_rng(0).choice(len(dirty_all), 5, replace=False)
fig, ax = plt.subplots(2, 5, figsize=(15, 6))
for j, i in enumerate(ids):
    ax[0, j].imshow(dirty_all[i], cmap='inferno'); ax[0, j].axis('off')
    ax[1, j].imshow(clean_all[i], cmap='inferno'); ax[1, j].axis('off')
ax[0, 0].set_title('dirty (noisy)', loc='left'); ax[1, 0].set_title('clean (GT)', loc='left')
plt.tight_layout(); plt.show()

## Data Quality Checks


In [ ]:
assert clean_all.shape == dirty_all.shape, 'shape mismatch!'
for nm, a in [('dirty', dirty_all), ('clean', clean_all)]:
    print(f'{nm}: range [{a.min():.4f}, {a.max():.4f}]  mean {a.mean():.4f}  std {a.std():.4f}  '
          f'NaNs {int(np.isnan(a).sum())}')
assert 0.0 <= dirty_all.min() and dirty_all.max() <= 1.0
assert 0.0 <= clean_all.min() and clean_all.max() <= 1.0
print(f'{len(clean_all)} paired samples, {clean_all.shape[1]}x{clean_all.shape[2]} px, both in [0,1]  ✓')

## Data Preprocessing
Training/evaluation use **per-patch min-max normalisation**: each 64×64 dirty patch is scaled to
`[0, 1]` by its own min/max, and the matching clean patch uses the *same* scale. This makes the
models robust to per-region intensity variation across the 600×600 field. (The diffusion model
additionally maps `[0,1] → [-1,1]` internally, the standard DDPM convention.)


## Dataset Classes
Single canonical `PatchDataset` (merged from the 4 identical copies across nb02–04).


In [ ]:
class PatchDataset(Dataset):
    \"\"\"One random PATCH_SIZE crop per image per epoch, per-patch min-max normalised to [0,1].\"\"\"
    def __init__(self, dirty, clean, indices, ps=CONFIG['PATCH_SIZE']):
        self.dirty, self.clean, self.indices, self.ps = dirty, clean, indices, ps
        self._h, self._w = dirty.shape[1], dirty.shape[2]
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        r = np.random.randint(0, self._h - self.ps + 1)
        c = np.random.randint(0, self._w - self.ps + 1)
        dp = self.dirty[idx, r:r+self.ps, c:c+self.ps].astype(np.float32)
        cp = self.clean[idx, r:r+self.ps, c:c+self.ps].astype(np.float32)
        lo, hi = dp.min(), dp.max()
        if hi > lo:
            dp = (dp - lo) / (hi - lo)
            cp = np.clip((cp - lo) / (hi - lo), 0.0, 1.0)
        return torch.from_numpy(dp[np.newaxis]), torch.from_numpy(cp[np.newaxis])
print('PatchDataset ready')

## DataLoaders
Split **once** (80/20, `random_state=42`); every stage reuses this split.


In [ ]:
seed_everything()
indices = np.arange(len(dirty_all))
train_idx, val_idx = train_test_split(indices, test_size=0.20, random_state=CONFIG['SEED'])
print('train', len(train_idx), '| val', len(val_idx))

train_loader = DataLoader(PatchDataset(dirty_all, clean_all, train_idx),
                          batch_size=CONFIG['BATCH_SIZE'], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(PatchDataset(dirty_all, clean_all, val_idx),
                          batch_size=CONFIG['BATCH_SIZE'], shuffle=False, num_workers=2, pin_memory=True)
print('train batches', len(train_loader), '| val batches', len(val_loader))

## Unified Evaluator (shared)
Merged from nb03 + nb04: full-image **sliding-window** inference (patch 64, stride 32, 50% overlap,
per-patch normalisation identical to training) + PSNR/SSIM/MSE. Used by the leaderboard below.


In [ ]:
@torch.no_grad()
def nn_denoise(model, img, use_sigmoid=False, is_vae=False):
    ps, stride, bs = CONFIG['PATCH_SIZE'], CONFIG['STRIDE'], CONFIG['BATCH_INFER']
    H, W = img.shape
    out_s = np.zeros((H, W), np.float64); out_c = np.zeros((H, W), np.float64)
    rows = list(range(0, H-ps+1, stride)); cols = list(range(0, W-ps+1, stride))
    if rows[-1]+ps < H: rows.append(H-ps)
    if cols[-1]+ps < W: cols.append(W-ps)
    patches, pos = [], []
    for r in rows:
        for c in cols:
            p = img[r:r+ps, c:c+ps].copy()
            lo, hi = p.min(), p.max()
            p = (p-lo)/(hi-lo) if hi > lo else p
            patches.append(p); pos.append((r, c))
    model.eval(); preds = []
    for s in range(0, len(patches), bs):
        b = torch.from_numpy(np.stack(patches[s:s+bs])[:, np.newaxis]).to(device)
        if is_vae:        o = model(b)[0]
        elif use_sigmoid:
            t = torch.zeros(b.size(0), dtype=torch.long, device=device); o = torch.sigmoid(model(b, t))
        else:             o = model(b)
        preds.extend(o.squeeze(1).cpu().numpy())
    for pred, (r, c) in zip(preds, pos):
        out_s[r:r+ps, c:c+ps] += pred; out_c[r:r+ps, c:c+ps] += 1.0
    return np.clip(out_s/np.maximum(out_c, 1e-8), 0, 1).astype(np.float32)

def calc(clean, denoised):
    d = np.clip(denoised.astype(np.float32), 0, 1)
    return (psnr_fn(clean, d, data_range=1.0),
            ssim_fn(clean, d, data_range=1.0),
            float(np.mean((clean - d)**2)))

LEADERBOARD = {}   # method -> (PSNR, SSIM, MSE)  filled across the notebook
print('evaluator ready')

# Classical Baselines (Week 1–2)
First milestone: establish a non-ML benchmark with Gaussian / Median / Wiener filtering, averaged
over 50 full 600×600 images. **Target for ML: beat Gaussian σ=2 (SSIM ≈ 0.52).**


In [ ]:
seed_everything()
res = run_baselines(clean_all, dirty_all, n_samples=50 if not CONFIG['QUICK_TEST'] else 10)
avg = {m: {k: float(np.mean(v)) for k, v in res[m].items()} for m in res}
print(f"\n{'Method':<14}{'PSNR':>9}{'SSIM':>9}{'MSE':>11}")
for m in avg:
    print(f"{m:<14}{avg[m]['PSNR']:>9.3f}{avg[m]['SSIM']:>9.4f}{avg[m]['MSE']:>11.6f}")

In [ ]:
methods = list(avg.keys())
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, key in zip(axes, ['PSNR', 'SSIM', 'MSE']):
    ax.bar(methods, [avg[m][key] for m in methods],
           color=['#555','#4C9EEB','#3BA55C','#E8715A','#F0B232'][:len(methods)])
    ax.set_title(key); ax.tick_params(axis='x', rotation=25)
plt.tight_layout(); plt.show()

**Finding (50 samples, 600×600):** Gaussian σ=2 is the best classical method — SSIM **0.521**,
MSE 0.006806. Median is sharp but lower SSIM; Wiener trails. These set the bar for the ML models.


# Autoencoder Section (Week 2–3)

▶ **Evolution.** *Previous:* none (first ML model). *New:* 3-level conv encoder–decoder + bottleneck,
no skip connections. *Why:* simplest learned denoiser to test whether a CNN can beat classical
filters. *Benefit:* learns disk structure; with the hybrid loss it surpasses the best classical
filter on SSIM.


### Architecture
Real source of `DenoisingAutoencoder` (`src/models/autoencoder.py`):


In [ ]:
print(inspect.getsource(DenoisingAutoencoder))
ae = DenoisingAutoencoder().to(device)
print('params:', f"{sum(p.numel() for p in ae.parameters()):,}")
with torch.no_grad():
    o = ae(torch.randn(2,1,64,64).to(device))
print('forward (2,1,64,64) ->', tuple(o.shape), 'range [%.3f, %.3f]' % (o.min(), o.max()))

### Training Procedure
Two experiments, preserved from nb02:
1. **MSE-only** (baseline).
2. **HybridLoss** `0.8·MSE + 0.2·(1−SSIM)` ("Tanmay's ratio").

▶ *Why the change:* MSE alone over-smooths; adding an SSIM term targets structure. ▶ *Benefit
(recorded):* Hybrid's MSE sub-component (0.007181) **beats** pure-MSE training (0.007239) — SSIM
regularisation improves pixel accuracy too.


In [ ]:
def train_recon(model, kind, loss_fn, epochs, tag):
    \"\"\"kind in {'ae','vae','unet'}; returns (train_hist, val_hist, best_val, best_state).\"\"\"
    model.to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=CONFIG['LR'])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5)
    ga = CONFIG['GRAD_ACCUM']
    best, best_state, tr_hist, va_hist = float('inf'), None, [], []
    print(f"{'ep':>3} {'train':>10} {'val':>10} {'lr':>9}")
    for ep in range(1, epochs+1):
        model.train(); opt.zero_grad(set_to_none=True); run = 0.0
        for step, (d, c) in enumerate(train_loader, 1):
            d, c = d.to(device), c.to(device)
            if kind == 'ae':   total = loss_fn(model(d), c)[0]
            elif kind == 'unet':
                t = torch.zeros(d.size(0), dtype=torch.long, device=device)
                total = loss_fn(torch.sigmoid(model(d, t)), c)[0]
            elif kind == 'vae':
                out, mu, lv = model(d); total = loss_fn(out, c, mu, lv)[0]
            (total/ga).backward(); run += total.item()*d.size(0)
            if step % ga == 0 or step == len(train_loader):
                opt.step(); opt.zero_grad(set_to_none=True)
        tr = run/len(train_loader.dataset)
        model.eval(); vr = 0.0
        with torch.no_grad():
            for d, c in val_loader:
                d, c = d.to(device), c.to(device)
                if kind == 'ae':   total = loss_fn(model(d), c)[0]
                elif kind == 'unet':
                    t = torch.zeros(d.size(0), dtype=torch.long, device=device)
                    total = loss_fn(torch.sigmoid(model(d, t)), c)[0]
                elif kind == 'vae':
                    out, mu, lv = model(d); total = loss_fn(out, c, mu, lv)[0]
                vr += total.item()*d.size(0)
        va = vr/len(val_loader.dataset); sched.step(va)
        tr_hist.append(tr); va_hist.append(va)
        if va < best:
            best, best_state = va, {k: v.clone() for k, v in model.state_dict().items()}
        if ep % 5 == 0 or ep == 1 or ep == epochs:
            print(f"{ep:>3} {tr:>10.6f} {va:>10.6f} {opt.param_groups[0]['lr']:>9.2e}"
                  f"{'  *' if va==best else ''}")
    if best_state: model.load_state_dict(best_state)
    return tr_hist, va_hist, best, best_state

def plot_curves(tr, va, title):
    plt.figure(figsize=(8,4.5)); e = range(1, len(tr)+1)
    plt.plot(e, tr, label='train', color='#4C9EEB', lw=2)
    plt.plot(e, va, label='val', color='#E8715A', lw=2, ls='--')
    b = int(np.argmin(va))+1
    plt.axvline(b, color='gray', ls=':'); plt.scatter([b],[min(va)], color='#E8715A', zorder=5)
    plt.xlabel('epoch'); plt.ylabel('loss'); plt.title(title); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
print('trainer ready')

In [ ]:
# Experiment 1: MSE-only autoencoder
seed_everything()
ae_mse = DenoisingAutoencoder().to(device)
mse_loss = nn.MSELoss()
class _MSEWrap:                      # adapt to (total, ...) interface
    def __call__(self, p, c): m = mse_loss(p, c); return (m, m, m)
tr, va, best_mse, _ = train_recon(ae_mse, 'ae', _MSEWrap(), CONFIG['EPOCHS_AE_MSE'], 'AE-MSE')
torch.save({'model_state_dict': ae_mse.state_dict(), 'val_loss': best_mse},
           f'{CKPT_DIR}/autoencoder_best.pth')
plot_curves(tr, va, 'Autoencoder — MSE only'); print('best val MSE:', round(best_mse,6))

In [ ]:
# Experiment 2: Hybrid-loss autoencoder
seed_everything()
ae = DenoisingAutoencoder().to(device)
tr, va, best_hyb, _ = train_recon(ae, 'ae', HybridLoss(0.8, 0.2), CONFIG['EPOCHS_AE_HYB'], 'AE-Hybrid')
torch.save({'model_state_dict': ae.state_dict(), 'val_loss': best_hyb, 'alpha':0.8, 'beta':0.2},
           f'{CKPT_DIR}/autoencoder_hybrid_best.pth')
plot_curves(tr, va, 'Autoencoder — Hybrid (0.8 MSE + 0.2 SSIM)'); print('best val hybrid:', round(best_hyb,6))

### Results (recorded, RTX 2050, 30 epochs)
| Variant | val metric | note |
|---|---|---|
| Noisy input | MSE 0.009048 | baseline |
| Gaussian σ=2 | MSE 0.006806 | best classical |
| AE MSE-only | MSE 0.007239 @ep26 | near-classical |
| **AE Hybrid** | MSE 0.007181 @ep27 | ✓ beats MSE-only; total hybrid 0.0344 |

### Analysis
The SSIM sub-loss drops 0.683→0.143 over 30 epochs (fast structural convergence). The hybrid model
becomes the **best single model by SSIM** in the unified evaluation (SSIM 0.7609).


In [ ]:
# Visual: dirty | clean | MSE-only | Hybrid (center 64x64 crop)
ae_mse.eval(); ae.eval()
ids = np.random.default_rng(7).integers(0, len(dirty_all), size=3); r=c=268
cols = ['dirty','clean GT','MSE-only','Hybrid']
fig, ax = plt.subplots(3, 4, figsize=(13, 9.5))
for k, t in enumerate(cols): ax[0,k].set_title(t)
for row, idx in enumerate(ids):
    dp = dirty_all[idx, r:r+64, c:c+64]; cp = clean_all[idx, r:r+64, c:c+64]
    lo, hi = dp.min(), dp.max(); dpn=(dp-lo)/(hi-lo) if hi>lo else dp; cpn=np.clip((cp-lo)/(hi-lo),0,1) if hi>lo else cp
    inp = torch.from_numpy(dpn[None,None]).to(device)
    with torch.no_grad():
        pm = ae_mse(inp).squeeze().cpu().numpy(); ph = ae(inp).squeeze().cpu().numpy()
    for k, im in enumerate([dpn, cpn, pm, ph]):
        ax[row,k].imshow(im, cmap='inferno', vmin=0, vmax=1); ax[row,k].axis('off')
plt.suptitle('Autoencoder: MSE-only vs Hybrid', fontweight='bold'); plt.tight_layout(); plt.show()

# VAE Section (Week 3)

▶ **Evolution.** *Previous:* deterministic AE (input → single bottleneck point). *New:* probabilistic
latent — encoder outputs μ and log σ² maps, sampled via the reparameterisation trick. *Why:* a
structured latent regularised toward N(0,I) generalises better and is reusable for future generative
conditioning. *Benefit:* competitive SSIM (0.706) with a smoother latent.


### Architecture
Real source of `DenoisingVAE` (`src/models/vae.py`):


In [ ]:
print(inspect.getsource(DenoisingVAE))
vae = DenoisingVAE(latent_dim=128).to(device)
print('params:', f"{sum(p.numel() for p in vae.parameters()):,}")
with torch.no_grad():
    out, mu, lv = vae(torch.randn(2,1,64,64).to(device))
print('out', tuple(out.shape), '| mu', tuple(mu.shape), '| log_var', tuple(lv.shape))

### Training Procedure
Loss = `0.8·MSE + 0.2·(1−SSIM) + 0.001·KL`, with **gradient accumulation** (×4 → effective batch 64)
to simulate a larger batch within 4 GB. KL = −0.5·mean(1 + log_var − μ² − exp(log_var)); the small
γ=0.001 lets reconstruction dominate early while preventing posterior collapse.


In [ ]:
seed_everything()
vae = DenoisingVAE(latent_dim=128).to(device)
tr, va, best_vae, _ = train_recon(vae, 'vae', VAELoss(0.8, 0.2, 0.001), CONFIG['EPOCHS_VAE'], 'VAE')
torch.save({'model_state_dict': vae.state_dict(), 'val_loss': best_vae,
            'latent_dim':128, 'alpha':0.8,'beta':0.2,'gamma':0.001}, f'{CKPT_DIR}/vae_best.pth')
plot_curves(tr, va, 'VAE — 0.8 MSE + 0.2 SSIM + 0.001 KL'); print('best val:', round(best_vae,6))

### Results & Analysis (recorded)
Best val loss **0.039160 @ ep23** (30-epoch run). The VAE adds a probabilistic, KL-regularised latent
over the AE; SSIM (0.7059) is just below AE-Hybrid but with better-behaved latent statistics
(analysed in the Visualization section). Good structural preservation; rings/gaps retained.


In [ ]:
# Visual: dirty | clean | Hybrid AE | VAE
ae.eval(); vae.eval()
ids = np.random.default_rng(99).integers(0, len(dirty_all), size=3); r=c=268
cols = ['dirty','clean GT','Hybrid AE','VAE']
fig, ax = plt.subplots(3, 4, figsize=(13, 9.5))
for k, t in enumerate(cols): ax[0,k].set_title(t)
for row, idx in enumerate(ids):
    dp = dirty_all[idx, r:r+64, c:c+64]; cp = clean_all[idx, r:r+64, c:c+64]
    lo, hi = dp.min(), dp.max(); dpn=(dp-lo)/(hi-lo) if hi>lo else dp; cpn=np.clip((cp-lo)/(hi-lo),0,1) if hi>lo else cp
    inp = torch.from_numpy(dpn[None,None]).to(device)
    with torch.no_grad():
        pa = ae(inp).squeeze().cpu().numpy(); pv = vae(inp)[0].squeeze().cpu().numpy()
    for k, im in enumerate([dpn, cpn, pa, pv]):
        ax[row,k].imshow(im, cmap='inferno', vmin=0, vmax=1); ax[row,k].axis('off')
plt.suptitle('VAE vs Hybrid AE', fontweight='bold'); plt.tight_layout(); plt.show()

# U-Net Section (Week 3)

▶ **Evolution.** *Previous:* AE/VAE have **no skip connections** — fine detail is lost through the
bottleneck. *New:* U-Net with encoder→decoder **skip connections**, residual blocks, GroupNorm and
sinusoidal **timestep** conditioning. *Why:* skips preserve high-frequency edges (ring boundaries);
the timestep input makes this the **same backbone** reused as the DDPM noise predictor. *Benefit:*
sharper reconstructions and a direct path to the diffusion model. For supervised denoising we set
`t = 0` and apply `sigmoid` to the (unbounded) output.


### Architecture
Real source of the U-Net (`src/models/unet.py` — `UNet` + `DenoisingUNet` preset):


In [ ]:
import src.models.unet as unet_mod
print(inspect.getsource(unet_mod.UNet))
print('\n# --- DenoisingUNet preset ---')
print(inspect.getsource(unet_mod.DenoisingUNet))
unet = DenoisingUNet(str(device))
print('params:', f"{sum(p.numel() for p in unet.parameters()):,}")
with torch.no_grad():
    o = unet(torch.randn(2,1,64,64).to(device), torch.zeros(2, dtype=torch.long, device=device))
print('forward ->', tuple(o.shape))

### Training Procedure
HybridLoss, Adam 1e-3, grad-accum ×4, `t=0`, `sigmoid` on outputs.


In [ ]:
seed_everything()
unet = DenoisingUNet(str(device))
tr, va, best_unet, _ = train_recon(unet, 'unet', HybridLoss(0.8, 0.2), CONFIG['EPOCHS_UNET'], 'UNet')
torch.save({'model_state_dict': unet.state_dict(), 'val_loss': best_unet,
            'alpha':0.8,'beta':0.2,'arch':'DenoisingUNet'}, f'{CKPT_DIR}/unet_best.pth')
plot_curves(tr, va, 'Supervised U-Net — Hybrid loss'); print('best val:', round(best_unet,6))

### Results & Analysis (recorded)
Best val hybrid **0.036899 @ ep23**. Architecture comparison:

| Model | Params | Skip | Probabilistic | DDPM backbone |
|---|---|:--:|:--:|:--:|
| Autoencoder | 1.73M | ❌ | ❌ | ❌ |
| VAE | ~3.2M | ❌ | ✅ | ❌ |
| **U-Net** | **3.4M** | ✅ | ❌ | ✅ |

Skip connections + residual blocks converge faster and preserve edges; the timestep embedding means
this exact network becomes ε_θ in the diffusion section.


In [ ]:
# Visual: clean | noisy | AE | VAE | U-Net  with per-patch SSIM under model columns
ae.eval(); vae.eval(); unet.eval()
ids = np.random.default_rng(99).choice(val_idx, size=3, replace=False); r=c=268
cols = ['Clean GT','Noisy','Autoencoder','VAE','U-Net']
fig, ax = plt.subplots(3, 5, figsize=(18, 11))
for k, t in enumerate(cols): ax[0,k].set_title(t, fontweight='bold')
for row, idx in enumerate(ids):
    dp = dirty_all[idx, r:r+64, c:c+64]; cp = clean_all[idx, r:r+64, c:c+64]
    lo, hi = dp.min(), dp.max(); dpn=(dp-lo)/(hi-lo) if hi>lo else dp; cpn=np.clip((cp-lo)/(hi-lo),0,1) if hi>lo else cp
    inp = torch.from_numpy(dpn[None,None]).to(device); t0 = torch.zeros(1, dtype=torch.long, device=device)
    with torch.no_grad():
        pa = ae(inp).squeeze().cpu().numpy(); pv = vae(inp)[0].squeeze().cpu().numpy()
        pu = torch.sigmoid(unet(inp, t0)).squeeze().cpu().numpy()
    for k, im in enumerate([cpn, dpn, pa, pv, pu]):
        ax[row,k].imshow(np.clip(im,0,1), cmap='inferno', vmin=0, vmax=1); ax[row,k].axis('off')
        if k >= 2:
            ax[row,k].set_xlabel(f'SSIM={ssim_fn(cpn, np.clip(im,0,1), data_range=1.0):.4f}')
plt.suptitle('Clean | Noisy | AE | VAE | U-Net (center 64x64)', fontweight='bold'); plt.tight_layout(); plt.show()

# Evaluation Section

### Metrics
- **MSE** — pixel error (lower better).
- **PSNR** = 10·log₁₀(1/MSE) dB (higher better) — favours smoothing.
- **SSIM** — structural similarity (higher better); **primary metric** here because it rewards
  preserving rings/gaps rather than blurring them away.

### Comparison Tables — 7-method leaderboard (full-image sliding window, 100 val samples)
Reproduces the Week-3 result. Classical filters run on full images; neural models use the shared
`nn_denoise` sliding window.


In [ ]:
seed_everything()
chosen = np.sort(np.random.default_rng(CONFIG['SEED']).choice(
    val_idx, size=min(CONFIG['N_EVAL_IMAGES'], len(val_idx)), replace=False))
print('evaluating', len(chosen), 'val images')

methods = ['Noisy input','Gaussian s=2','Median 3x3','Wiener',
           'AE MSE-only','AE HybridLoss','VAE (MSE+SSIM+KL)','U-Net HybridLoss']
acc = {m: [[],[],[]] for m in methods}
t0 = time.time()
for n, idx in enumerate(chosen, 1):
    c, d = clean_all[idx], dirty_all[idx]
    def push(name, arr):
        p, s, m = calc(c, arr)
        acc[name][0].append(p); acc[name][1].append(s); acc[name][2].append(m)
    push('Noisy input', d)
    push('Gaussian s=2', gaussian_filter(d, sigma=2.0))
    push('Median 3x3', median_filter(d, size=3))
    push('Wiener', wiener_filter(d).astype(np.float32))
    push('AE MSE-only', nn_denoise(ae_mse, d))
    push('AE HybridLoss', nn_denoise(ae, d))
    push('VAE (MSE+SSIM+KL)', nn_denoise(vae, d, is_vae=True))
    push('U-Net HybridLoss', nn_denoise(unet, d, use_sigmoid=True))
    if n % 25 == 0: print(f'  [{n}/{len(chosen)}] {time.time()-t0:.0f}s')

for m in methods:
    LEADERBOARD[m] = (float(np.mean(acc[m][0])), float(np.mean(acc[m][1])), float(np.mean(acc[m][2])))
df = (pd.DataFrame([(m,*LEADERBOARD[m]) for m in methods], columns=['Method','PSNR','SSIM','MSE'])
        .sort_values('SSIM', ascending=False).reset_index(drop=True))
df.index += 1
df.to_csv(f'{OUT_DIR}/metrics_leaderboard.csv', index_label='Rank')
print(df.round({'PSNR':4,'SSIM':4,'MSE':6}).to_string())

**Finding:** neural models dominate SSIM (AE-Hybrid **0.76**, VAE 0.71, U-Net 0.70) — +80% over the
best classical filter — while classical filters win PSNR/MSE by blurring. For disk science SSIM is
what matters, so the ML models are the real winners.


# Diffusion Model Section (Week 4)

▶ **Evolution.** *Previous:* one-shot regression (AE/VAE/U-Net) — predict clean directly. *New:* a
**conditional DDPM** — learn to predict the noise added to the clean image, conditioned on the dirty
observation, then iteratively denoise with **DDIM**. *Why:* diffusion models capture the full
conditional distribution and can produce sharper, more detailed reconstructions. *Benefit:* a
principled generative denoiser; the U-Net backbone (above) is reused as ε_θ.

The model was scaled for a 4 GB RTX 2050 (`ch=64`, 4 levels, ~17.2M params) and trained with EMA +
multi-GPU `DataParallel`. The local 40-epoch run was under-trained (SSIM 0.235) — hence the move to
Kaggle for many more epochs.


### Noise Scheduler
Real source of `NoiseScheduler` (`src/models/noise_scheduler.py`) + the DDPM β-schedule used by the runner:


In [ ]:
print(inspect.getsource(NoiseScheduler))
print('\n# --- beta schedule used by the conditional DDPM runner ---')
print(inspect.getsource(diffusion_mod.get_beta_schedule))

### Forward Diffusion
`q(x_t|x_0)`: `x_t = √ᾱ_t·x_0 + √(1−ᾱ_t)·ε`. The training objective regresses the model's predicted
noise onto the true ε (conditioned on the dirty image, channel-concatenated):


In [ ]:
print(inspect.getsource(diffusion_mod.noise_estimation_loss))

### Reverse Diffusion
DDIM sampling (`generalized_steps`) — deterministic reverse process conditioned on the dirty image:


In [ ]:
print(inspect.getsource(diffusion_mod.generalized_steps))

### DDPM Architecture
Real source of the scaled `DiffusionUNet` (`src/models/diffusion_unet.py`) and its config:


In [ ]:
print(inspect.getsource(default_diffusion_config))
print('# DiffusionUNet (attention + timestep-conditioned ResNet blocks) — see src/models/diffusion_unet.py')
print('class signature:', str(inspect.signature(DiffusionUNet.__init__)))
cfg = default_diffusion_config(image_size=CONFIG['PATCH_SIZE'])
print('scaled config:', dict(cfg['model']))

### Training Loop
2-channel `[dirty, clean]` patches via `create_dataloaders`; `DenoisingDiffusion` handles EMA, antithetic timestep sampling, multi-GPU, and checkpointing.


In [ ]:
seed_everything()
d_tr, c_tr = dirty_all[train_idx], clean_all[train_idx]
d_va, c_va = dirty_all[val_idx],   clean_all[val_idx]
dif_train, dif_val = create_dataloaders(
    dirty_train=d_tr, clean_train=c_tr, dirty_val=d_va, clean_val=c_va,
    batch_size=CONFIG['DIFF_BATCH_IMG'], num_workers=2,
    parse_patches=True, patch_size=CONFIG['PATCH_SIZE'], n_patches=CONFIG['DIFF_PATCH_N'])

DIFF_CKPT = f'{CKPT_DIR}/diffusion_best.pth.tar'
diffusion = DenoisingDiffusion(config=cfg, device=str(device), lr=CONFIG['DIFF_LR'],
                               checkpoint_path=DIFF_CKPT)
print('DDPM params:', f"{sum(p.numel() for p in diffusion._core.parameters()):,}",
      '| GPUs:', diffusion.num_gpus, '(DataParallel)' if diffusion.data_parallel else '(single)')

### Resume Training
If a checkpoint exists and `CONFIG['RESUME_DIFF']`, resume from it; otherwise train fresh. `load_checkpoint` restores model+EMA+optimizer and sets `start_epoch`.


In [ ]:
if CONFIG['RESUME_DIFF'] and os.path.exists(DIFF_CKPT):
    diffusion.load_checkpoint(DIFF_CKPT)
    print('resumed from', DIFF_CKPT, '| start_epoch', diffusion.start_epoch)
res = diffusion.train(dif_train, dif_val, n_epochs=CONFIG['EPOCHS_DIFF'], log_every_step=10)
print('best val noise-loss:', round(res['best_val_loss'], 4))

### Validation Loop & loss curve
(Validation noise-MSE is computed each epoch inside `train`; plotted here.)


In [ ]:
plot_curves(res['train_losses'], res['val_losses'], 'Conditional DDPM — noise-estimation loss')

### Checkpointing
Best checkpoint (model + EMA + optimizer + config) is saved automatically to `/kaggle/working` and is portable to single-GPU.


In [ ]:
diffusion.load_checkpoint(DIFF_CKPT)
print('checkpoint:', DIFF_CKPT, '| size %.0f MB' % (os.path.getsize(DIFF_CKPT)/1e6))

### Inference (DDIM)
Denoise validation patches with 25 DDIM steps (EMA weights); report PSNR/SSIM/MSE.


In [ ]:
m = diffusion.evaluate(dif_val, sampling_timesteps=CONFIG['DDIM_STEPS'],
                       max_batches=None if CONFIG['QUICK_TEST'] else 12, use_ema=True)
print(f"DDPM (patch eval)  n={m['n']}  PSNR={m['psnr']:.4f}  SSIM={m['ssim']:.4f}  MSE={m['mse']:.6f}")

In [ ]:
# Visual: dirty -> DDPM -> clean
x, _ = next(iter(dif_val)); x = x.flatten(0,1) if x.ndim == 5 else x
dp6 = x[:6, 0:1].clamp(0,1); cp6 = x[:6, 1:2].clamp(0,1)
pp6 = diffusion.sample(dp6, sampling_timesteps=CONFIG['DDIM_STEPS'], use_ema=True).cpu()
fig, ax = plt.subplots(3, 6, figsize=(15, 7.5))
for i,(im,nm) in enumerate([(dp6,'dirty'),(pp6,'DDPM'),(cp6,'clean')]):
    for j in range(6): ax[i,j].imshow(im[j,0], cmap='inferno'); ax[i,j].axis('off')
    ax[i,0].set_title(nm, loc='left')
plt.tight_layout(); plt.show()

# Diffusion Model — Week-4 Scaled Upgrade (Kaggle T4×2)

▶ **Evolution.** *Previous:* the conditional DDPM above used a 4 GB-constrained backbone
(`ch=64`, 4 levels, ~17.2M params, 64×64 patches) — fine for an RTX 2050 but under-capacity, and
the local 40-epoch run only reached SSIM ≈ 0.235. *New:* now that training has moved to Kaggle's
**T4×2 (2×16 GB)**, scale the model and the receptive field up:

| | Baseline (above) | **Scaled (this section)** |
|---|---|---|
| `ch` | 64 | **128** |
| `ch_mult` | [1,2,2,4] (4 levels) | **[1,1,2,2,4,4] (6 levels)** |
| attention | @16 | @16 |
| params | 17.2M | **~110M (plan's 51M-style 6-level config)** |
| patch resolution | 64×64 | **128×128** (4× the area) |
| GPUs | 1 | **2 (DataParallel)** |

*Why:* more capacity + a 4× larger field-of-view per patch captures more disk structure per step;
T4×2 has the VRAM for it. *Benefit:* a stronger conditional denoiser, the proper Week-4 deliverable
(loss curve + first sample outputs).

> **Note on "full 6-level / 51M / 600×600".** A 6-level U-Net needs the input side divisible by
> 2⁵=32; 600 is not (600/32=18.75), so full-resolution 6-level won't even build, and a 110M model at
> 600² overflows 16 GB. We therefore use a **6-level ch=128 model on 128×128 patches** — the
> recommended middle ground between the earlier 64×64 and full size. The baseline above is **kept**
> as the documented starting point so the scaling story stays in the record.


### Forward Diffusion — q(xₜ | x₀)
The forward process adds Gaussian noise progressively over **T=1000** timesteps:
`xₜ = √ᾱₜ · x₀ + √(1−ᾱₜ) · ε`, `ε ~ N(0, I)`. As t→T the image becomes pure noise. Below we
visualise it on a real clean 128×128 patch using the repo's `NoiseScheduler.q_sample`.


In [ ]:
SCALE_RES = 128   # scaled-DDPM patch resolution

# one clean 128x128 patch (per-patch min-max, same convention as training)
seed_everything()
_pi = int(val_idx[0])
_r = (dirty_all.shape[1] - SCALE_RES)//2; _c = (dirty_all.shape[2] - SCALE_RES)//2
_cp = clean_all[_pi, _r:_r+SCALE_RES, _c:_c+SCALE_RES].astype(np.float32)
_lo,_hi = _cp.min(), _cp.max(); _cp = (_cp-_lo)/(_hi-_lo) if _hi>_lo else _cp
x0 = torch.from_numpy(_cp[None,None]).float()

sched_lin = NoiseScheduler(timesteps=1000, beta_schedule='linear')
steps = [0, 250, 500, 750, 999]   # plan Task 1.2
fig, ax = plt.subplots(1, len(steps), figsize=(16, 3))
for k, t in enumerate(steps):
    xt, _ = sched_lin.q_sample(x0, torch.tensor([t]))
    ax[k].imshow(xt[0,0].numpy(), cmap='inferno'); ax[k].axis('off'); ax[k].set_title(f't={t}')
plt.suptitle('Forward diffusion q(xₜ|x₀) — progressive noising (linear schedule)', fontweight='bold')
plt.tight_layout(); plt.show()

### Noise Scheduler — linear vs cosine β-schedule
The schedule controls how fast ᾱₜ (signal retention) decays. **Linear** noises quickly; **cosine**
(Nichol & Dhariwal 2021) keeps more signal in the middle of the trajectory, which usually helps.
Both live in `src/models/noise_scheduler.py` (this is the scheduler from PR #22).


In [ ]:
sched_cos = NoiseScheduler(timesteps=1000, beta_schedule='cosine')
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(sched_lin.betas.numpy(), label='linear'); ax[0].plot(sched_cos.betas.numpy(), label='cosine')
ax[0].set_title('βₜ'); ax[0].set_xlabel('t'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(sched_lin.alphas_cumprod.numpy(), label='linear'); ax[1].plot(sched_cos.alphas_cumprod.numpy(), label='cosine')
ax[1].set_title('ᾱₜ (cumulative signal)'); ax[1].set_xlabel('t'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.suptitle('Beta schedules — linear vs cosine', fontweight='bold'); plt.tight_layout(); plt.show()

# which schedule the scaled DDPM trains with (the DenoisingDiffusion runner uses the same names)
SCALE_BETA_SCHEDULE = 'linear'   # set to 'cosine' to try the cosine schedule
print('scaled DDPM beta schedule:', SCALE_BETA_SCHEDULE)

### Scaled config + data (ch=128, 5-level, 128×128)
Conditional DDPM: input is the channel concat `[dirty, x_t]`, target is the noise ε. Patches are the
2-channel `[dirty, clean]` tensors produced by `create_dataloaders`.


In [ ]:
scfg = default_diffusion_config(image_size=SCALE_RES)
scfg.model.ch = 128
scfg.model.ch_mult = [1, 1, 2, 2, 4, 4]       # 6 levels: 128->64->32->16->8->4 (attn @16); ~plan's 51M config
scfg.model.attn_resolutions = [16]
scfg.diffusion.beta_schedule = SCALE_BETA_SCHEDULE

CONFIG['SCALE_BATCH_IMG'] = 4    # images/batch; x SCALE_PATCH_N patches = effective batch
CONFIG['SCALE_PATCH_N']   = 2    # 128px patches per image  -> effective 8 patches/step
CONFIG['EPOCHS_DIFF_BIG'] = CONFIG['EPOCHS_DIFF'] if not CONFIG['QUICK_TEST'] else 2

seed_everything()
sd_tr, sd_va = create_dataloaders(
    dirty_train=dirty_all[train_idx], clean_train=clean_all[train_idx],
    dirty_val=dirty_all[val_idx],     clean_val=clean_all[val_idx],
    batch_size=CONFIG['SCALE_BATCH_IMG'], num_workers=2,
    parse_patches=True, patch_size=SCALE_RES, n_patches=CONFIG['SCALE_PATCH_N'])
print('scaled config:', dict(scfg['model']))
print('train batches', len(sd_tr), '| val batches', len(sd_va))

### Build scaled runner — with automatic OOM fallback
T4 VRAM for a 71M model at 128×128 could not be verified offline, so this cell **probes** one
forward+backward step and, on CUDA OOM, automatically backs off: halve the per-image batch, then
drop to 64×64 patches. Whatever survives is what training uses. Check the printout on Kaggle.


In [ ]:
def build_scaled_runner():
    res = SCALE_RES
    for attempt in range(4):
        cfg_try = default_diffusion_config(image_size=res)
        cfg_try.model.ch = 128
        cfg_try.model.ch_mult = [1,1,2,2,4,4] if res >= 128 else [1,2,2,4]
        cfg_try.model.attn_resolutions = [16]
        cfg_try.diffusion.beta_schedule = SCALE_BETA_SCHEDULE
        run = DenoisingDiffusion(config=cfg_try, device=str(device), lr=CONFIG['DIFF_LR'],
                                 checkpoint_path=f'{CKPT_DIR}/diffusion_scaled_best.pth.tar')
        try:
            loader, _ = create_dataloaders(
                dirty_train=dirty_all[train_idx], clean_train=clean_all[train_idx],
                batch_size=CONFIG['SCALE_BATCH_IMG'], num_workers=0,
                parse_patches=True, patch_size=res, n_patches=CONFIG['SCALE_PATCH_N'])
            xb, _ = next(iter(loader)); xb = xb.flatten(0,1) if xb.ndim==5 else xb
            xb = (2*xb-1).to(device)
            e = torch.randn_like(xb[:, 1:]); t = torch.randint(0, run.num_timesteps, (xb.size(0),), device=device)
            loss = diffusion_mod.noise_estimation_loss(run.model, xb, t, e, run.betas)
            loss.backward(); run.optimizer.zero_grad()
            print(f'[OK] scaled runner: {res}x{res}, batch_img={CONFIG["SCALE_BATCH_IMG"]}, '
                  f'{sum(p.numel() for p in run._core.parameters()):,} params, '
                  f'GPUs={run.num_gpus}{" DataParallel" if run.data_parallel else ""}')
            return run, res
        except RuntimeError as ex:
            if 'out of memory' not in str(ex).lower(): raise
            torch.cuda.empty_cache(); del run
            if CONFIG['SCALE_BATCH_IMG'] > 1:
                CONFIG['SCALE_BATCH_IMG'] //= 2; print(f'[OOM] -> retry batch_img={CONFIG["SCALE_BATCH_IMG"]}')
            elif res > 64:
                res = 64; print(f'[OOM] -> fall back to {res}x{res} patches')
            else:
                raise
    raise RuntimeError('could not fit scaled DDPM even at minimum settings')

scaled_diffusion, SCALE_RES_USED = build_scaled_runner()
# rebuild loaders to match whatever fit
sd_tr, sd_va = create_dataloaders(
    dirty_train=dirty_all[train_idx], clean_train=clean_all[train_idx],
    dirty_val=dirty_all[val_idx],     clean_val=clean_all[val_idx],
    batch_size=CONFIG['SCALE_BATCH_IMG'], num_workers=2,
    parse_patches=True, patch_size=SCALE_RES_USED, n_patches=CONFIG['SCALE_PATCH_N'])
print('training at', SCALE_RES_USED, 'px')

### Train the scaled DDPM
Best checkpoint (model + EMA + optimizer + config) is saved to `diffusion_scaled_best.pth.tar`. Set `CONFIG['EPOCHS_DIFF_BIG']` for your time budget.


In [ ]:
scaled_res = scaled_diffusion.train(sd_tr, sd_va, n_epochs=CONFIG['EPOCHS_DIFF_BIG'], log_every_step=10)
print('scaled DDPM best val noise-loss:', round(scaled_res['best_val_loss'], 4))

### Loss curve — DDPM training deliverable


In [ ]:
plot_curves(scaled_res['train_losses'], scaled_res['val_losses'],
            f'Scaled conditional DDPM (ch=128, 5-level, {SCALE_RES_USED}px) — noise-estimation loss')

### First sample outputs — DDIM denoising (deliverable)
Dirty → scaled-DDPM (DDIM, EMA weights) → clean, on held-out validation patches. Plus PSNR/SSIM/MSE.


In [ ]:
scaled_diffusion.load_checkpoint(f'{CKPT_DIR}/diffusion_scaled_best.pth.tar')
sm = scaled_diffusion.evaluate(sd_va, sampling_timesteps=CONFIG['DDIM_STEPS'],
                               max_batches=None if CONFIG['QUICK_TEST'] else 8, use_ema=True)
print(f"Scaled DDPM ({SCALE_RES_USED}px)  n={sm['n']}  PSNR={sm['psnr']:.4f}  SSIM={sm['ssim']:.4f}  MSE={sm['mse']:.6f}")

xb, _ = next(iter(sd_va)); xb = xb.flatten(0,1) if xb.ndim==5 else xb
dp = xb[:5, 0:1].clamp(0,1); cp = xb[:5, 1:2].clamp(0,1)
pp = scaled_diffusion.sample(dp, sampling_timesteps=CONFIG['DDIM_STEPS'], use_ema=True).cpu()
fig, ax = plt.subplots(3, 5, figsize=(15, 9))
for i,(im,nm) in enumerate([(dp,'dirty'),(pp,'DDPM (scaled)'),(cp,'clean')]):
    for j in range(5): ax[i,j].imshow(im[j,0], cmap='inferno'); ax[i,j].axis('off')
    ax[i,0].set_title(nm, loc='left', fontsize=12)
plt.suptitle(f'First sample outputs — scaled DDPM @ {SCALE_RES_USED}px', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/diffusion_scaled_samples.png', dpi=130); plt.show()

### Scaled DDPM — notes
- This is the Week-4 deliverable: **loss curve + first sample outputs** for a larger conditional DDPM.
- Like any DDPM, sample quality keeps improving with many more epochs — raise `EPOCHS_DIFF_BIG`.
- The 64px/17M baseline above is retained for comparison; the unified patch table (Visualization
  section) still reports the **baseline** DDPM. To rank the scaled model there, swap `diffusion` →
  `scaled_diffusion` in the `UNIFIED['DDPM']` line (resolutions differ, so compare with that in mind).


# Visualization Section


### Latent Space Analysis (VAE)
μ should sit near 0 and per-dimension KL should be small but non-zero (no posterior collapse).


In [ ]:
vae.eval()
with torch.no_grad():
    d, _ = next(iter(val_loader)); _, mu, lv = vae(d.to(device))
mu_np = mu.cpu().numpy().reshape(-1); lv_np = lv.cpu().numpy()
kl_per_dim = (-0.5*(1 + lv_np - mu.cpu().numpy()**2 - np.exp(lv_np))).mean(axis=(0,2,3))
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(mu_np, bins=60, color='#4C9EEB'); ax[0].set_title('latent mu (≈0)')
ax[1].hist(np.exp(0.5*lv_np).reshape(-1), bins=60, color='#3BA55C'); ax[1].set_title('latent sigma')
ax[2].plot(np.sort(kl_per_dim)[::-1], color='#E8715A'); ax[2].set_title('per-dim KL (sorted)')
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('mean|mu| %.3f  mean sigma %.3f  active dims (KL>1e-3): %d/%d'
      % (np.abs(mu_np).mean(), np.exp(0.5*lv_np).mean(), int((kl_per_dim>1e-3).sum()), kl_per_dim.size))

### Model Comparisons — all five approaches on identical held-out patches
A fair, single-protocol comparison: every method (classical + AE + VAE + U-Net + **DDPM**) is scored on the *same* normalized 64×64 validation patches.


In [ ]:
seed_everything()
np.random.seed(123)
ev = PatchDataset(dirty_all, clean_all, np.resize(val_idx, CONFIG['N_EVAL_PATCHES']))
dirty_eval = torch.stack([ev[i][0] for i in range(CONFIG['N_EVAL_PATCHES'])])
clean_eval = torch.stack([ev[i][1] for i in range(CONFIG['N_EVAL_PATCHES'])])

def patch_metrics(pred):
    pred = pred.clamp(0,1); cl = clean_eval.clamp(0,1)
    mse = torch.mean((pred-cl)**2, dim=(1,2,3))
    psnr = 10*torch.log10(1.0/torch.clamp(mse, min=1e-10))
    s = ssim_torch(pred, cl, data_range=1.0, size_average=False)
    return float(psnr.mean()), float(s.mean()), float(mse.mean())

@torch.no_grad()
def model_pred(model, kind, bs=32):
    out = []
    for i in range(0, len(dirty_eval), bs):
        d = dirty_eval[i:i+bs].to(device)
        if kind=='ae': p = model(d)
        elif kind=='vae': p = model(d)[0]
        elif kind=='unet':
            t = torch.zeros(d.size(0), dtype=torch.long, device=device); p = torch.sigmoid(model(d,t))
        out.append(p.cpu())
    return torch.cat(out)

UNIFIED = {}
de = dirty_eval.numpy()
for nm, fn in [('Noisy', lambda x:x), ('Gaussian s2', lambda x:gaussian_denoise(x,2.0)),
               ('Median 3x3', lambda x:median_denoise(x,3)), ('Wiener', lambda x:np.nan_to_num(wiener_denoise(x)))]:
    pr = torch.from_numpy(np.stack([fn(de[k,0])[None] for k in range(len(de))])).float()
    UNIFIED[nm] = patch_metrics(pr)
UNIFIED['Autoencoder'] = patch_metrics(model_pred(ae,'ae'))
UNIFIED['VAE']         = patch_metrics(model_pred(vae,'vae'))
UNIFIED['U-Net']       = patch_metrics(model_pred(unet,'unet'))
ddpm_pred = torch.cat([diffusion.sample(dirty_eval[i:i+32], sampling_timesteps=CONFIG['DDIM_STEPS'], use_ema=True).cpu()
                       for i in range(0, len(dirty_eval), 32)])
UNIFIED['DDPM']        = patch_metrics(ddpm_pred)

udf = (pd.DataFrame([(k,*v) for k,v in UNIFIED.items()], columns=['Method','PSNR','SSIM','MSE'])
         .sort_values('SSIM', ascending=False).reset_index(drop=True))
udf.to_csv(f'{OUT_DIR}/metrics_unified_patch.csv', index=False)
print(udf.round({'PSNR':4,'SSIM':4,'MSE':6}).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].barh(udf['Method'], udf['SSIM']); ax[0].invert_yaxis(); ax[0].set_title('SSIM (higher better)')
ax[1].barh(udf['Method'], udf['PSNR']); ax[1].invert_yaxis(); ax[1].set_title('PSNR dB')
for a in ax: a.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

# Final Pipeline

### Model Loading & Prediction Pipeline
End-to-end: load any trained checkpoint and denoise a full 600×600 image with one call.


In [ ]:
def load_model(kind):
    if kind == 'ae':
        m = DenoisingAutoencoder().to(device)
        m.load_state_dict(torch.load(f'{CKPT_DIR}/autoencoder_hybrid_best.pth', map_location=device)['model_state_dict'])
    elif kind == 'vae':
        m = DenoisingVAE(latent_dim=128).to(device)
        m.load_state_dict(torch.load(f'{CKPT_DIR}/vae_best.pth', map_location=device)['model_state_dict'])
    elif kind == 'unet':
        m = DenoisingUNet(str(device))
        m.load_state_dict(torch.load(f'{CKPT_DIR}/unet_best.pth', map_location=device)['model_state_dict'])
    else:
        raise ValueError(kind)
    m.eval(); return m

def denoise_image(img, method='ae'):
    \"\"\"Denoise a full (H,W) image in [0,1]. method in {'ae','vae','unet','ddpm'}.\"\"\"
    img = img.astype(np.float32)
    if method == 'ddpm':
        # DDIM over the same sliding window, per-patch
        ps, stride = CONFIG['PATCH_SIZE'], CONFIG['PATCH_SIZE']  # non-overlap for speed
        H, W = img.shape; out = np.zeros_like(img)
        for r in range(0, H-ps+1, stride):
            for c in range(0, W-ps+1, stride):
                p = img[r:r+ps, c:c+ps]; lo, hi = p.min(), p.max()
                pn = (p-lo)/(hi-lo) if hi > lo else p
                pred = diffusion.sample(torch.from_numpy(pn[None,None]).float(),
                                        sampling_timesteps=CONFIG['DDIM_STEPS'], use_ema=True)[0,0].cpu().numpy()
                out[r:r+ps, c:c+ps] = pred
        return np.clip(out, 0, 1)
    mdl = load_model(method)
    return nn_denoise(mdl, img, use_sigmoid=(method=='unet'), is_vae=(method=='vae'))

# Demo on one validation image
demo_idx = int(val_idx[0]); dimg = dirty_all[demo_idx]; cimg = clean_all[demo_idx]
fig, ax = plt.subplots(1, 5, figsize=(20, 4))
ax[0].imshow(dimg, cmap='inferno'); ax[0].set_title('dirty')
for k, meth in enumerate(['ae','vae','unet'], start=1):
    out = denoise_image(dimg, meth)
    ax[k].imshow(out, cmap='inferno'); ax[k].set_title(f'{meth}  SSIM={ssim_fn(cimg,out,data_range=1.0):.3f}')
ax[4].imshow(cimg, cmap='inferno'); ax[4].set_title('clean GT')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()
print('Pipeline ready: denoise_image(img, method) for ae/vae/unet/ddpm')

# Conclusion

### Key Findings
- **SSIM is the right metric.** Classical filters win PSNR/MSE by blurring, but neural models win
  SSIM by **+80%**, preserving the disk rings/gaps that matter scientifically.
- **Best single model: Autoencoder + Hybrid loss** (SSIM ≈ 0.761 on the 100-sample leaderboard),
  narrowly ahead of VAE (0.706) and U-Net (0.704).
- **Hybrid loss helps pixel accuracy too** — its MSE component (0.007181) beat pure-MSE training
  (0.007239); SSIM regularisation is a net win.
- **Diffusion is promising but data/compute-hungry** — the local 40-epoch DDPM was under-trained
  (SSIM 0.235). Long Kaggle training (this notebook, multi-GPU) is the path to competitive samples.

### Lessons Learned
- Per-patch min-max normalisation is essential for stable training across the variable-intensity field.
- Gradient accumulation buys an effective batch of 64 within 4 GB.
- For DDPMs, low noise-prediction loss ≠ good samples — sample quality needs many more steps.
- Keeping the U-Net timestep-conditioned from the start let the same backbone serve both supervised
  denoising and the DDPM.

### Future Work
- Train the DDPM for several hundred epochs on Kaggle (T4×2) and re-evaluate vs the leaderboard.
- Try a larger DDPM (`ch=128` / deeper) using the 16 GB headroom.
- Overlapping-patch DDIM for seamless full-image diffusion denoising.
- DDIM step / guidance-scale sweeps; classifier-free conditioning.
- Validate on real ALMA observations.


## Phase 6 — Validation Report

1. **Missing dependencies** — `pytorch-msssim`, `torchinfo` are pip-installed by the bootstrap
   (Internet must be ON). All else (torch, numpy, scipy, skimage, sklearn, pandas, matplotlib) ships
   on the Kaggle image.
2. **Potential bugs / gotchas** — Internet OFF → clone/pip fail; data not attached → `find` warns and
   `np.load` fails (attach the Dataset). Wiener can emit NaNs → guarded with `nan_to_num`. Re-running
   training cells overwrites checkpoints (intended).
3. **Conflicting implementations resolved** — diffusion needs 2-channel `[-1,1]` patches vs the
   single-channel `[0,1]` patches of AE/VAE/U-Net; built as separate loaders so neither clobbers the
   other. Two eval protocols are kept on purpose: full-image sliding-window leaderboard (reproduces
   Week-3 numbers) and a shared-patch unified table (fair incl. DDPM) — clearly labelled.
4. **Duplicate code removed** — 5 bootstraps→1, 4 `PatchDataset`→1, 2 sliding-window evaluators→1,
   nb02 cell-30 `__main__` script dropped (re-implemented training; logic preserved in `train_recon`).
5. **Manual review** — set `CONFIG['EPOCHS_DIFF']` for your time budget; optionally scale the DDPM
   (`cfg.model.ch`); verify your Kaggle Dataset slug is discovered under `/kaggle/input`.
6. **Estimated runtime (T4×2)** — AE×2 + VAE + U-Net ≈ 10–20 min total; DDPM ≈ 0.4–0.8 min/epoch
   (≈ 1–2 h at 150 epochs); evaluations ≈ 5–10 min. `QUICK_TEST=True` finishes end-to-end in a few min.
7. **Memory** — data ~4.1 GB RAM; GPU peak ≈ 2–4 GB (DDPM batch 64 patches across 2 GPUs). Fits T4
   (16 GB) comfortably; AE/VAE/U-Net also fit a 4 GB card.
8. **GPU recommendation** — **T4×2** (uses DataParallel for the DDPM) or P100; CPU works for the
   classical baselines only.
